# Part 4: Work IQ で職場のコンテキストを追加する

受信箱を開くと、緊急のメールが 3 通届いていました。Seattle ストアで Professional Claw Hammer が品切れになり、顧客からはクレームが届き、同僚たちはあなたに在庫の振り替えや緊急補充の調整を依頼してきています。メールスレッドが何を言っているかを把握しつつ、緊急調達のエスカレーションについて会社のハンドブックがどう定めているかを調べる必要があります。

**🎯 ミッション**
- 実際の Microsoft 365 データ（メール、Teams、カレンダー）をクエリできる **Work IQ knowledge source** を追加する
- 3 ソースの knowledge base（HR docs + health docs + Work IQ）を構築する
- 片方のサブ回答が受信箱から、もう片方が会社ポリシーから返る質問を投げる

## 構成要素

Part 1〜3 では knowledge base がドキュメントや構造化データをクエリしていました。ここでは、あなた個人の職場コンテキストにつながるソースを追加します。

- **Work IQ Knowledge Source**: Microsoft 365（Outlook、Teams、Calendar、OneDrive）に接続します。KB は実行時にこのソースをクエリし、ユーザーの質問に関連するメール、チャット、カレンダーイベントを取り出します。
- **Delegated token**: このノートブックでは `InteractiveBrowserCredential` で現在のユーザーをサインインさせ、Azure AI Search の委任トークンを取得します。Search は Work IQ のような保護されたソースをクエリする際に、その委任ユーザー ID を使用します。

## Step 1: 環境変数をセットアップする

Azure AI Search と Azure OpenAI のリソース設定を読み込みます。プロジェクトフォルダーの `.env` ファイルにこのパートで必要な値が入っています。

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## Step 2: サインインと delegated token の取得

knowledge base の retrieval で Work IQ にあなたの ID を使うには、このテナント内の Microsoft 365 データへのアクセス権を持つサインイン済みユーザーの OAuth トークンが必要です。

下のセルを実行し、手順サイドバーに記載された資格情報で Azure にサインインしてください。成功すると Azure AI Search スコープの委任トークンを取得し、retrieval 時に `query_source_authorization` として使用するために保存します。

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## Step 3: 3 つの knowledge source を作成する

この knowledge base 用に、これまでと同じ 2 つの search-index ソースに加え、Work IQ の knowledge source を追加して計 3 つを作成します。

* `hrdocs-knowledge-source`: `hrdocs` 検索インデックスを参照
* `healthdocs-knowledge-source`: `healthdocs` 検索インデックスを参照
* `workiq-knowledge-source`: 職場コンテキストのために Work IQ を参照

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


## Step 4: マルチソース + Work IQ の knowledge base を作成する

knowledge base は knowledge source 群を LLM と、retrieval の振る舞いに関する指示と組み合わせます。

このノートブックでは `outputMode` に `answerSynthesis` を指定し、アタッチしたモデルが質問に直接答えられるようにします。`retrievalReasoningEffort` は `low` に設定し、クエリプランニングと source 選択にモデルを利用します。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base combining indexed company documents and Work IQ",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context (emails, chats, events, meetings, etc). Use the search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")


## Step 5: knowledge base にクエリを投げる

2 つの観点を含む質問を投げます。片方は Work IQ からあなた自身の受信箱コンテキストが必要で、もう片方は HR インデックスから会社ポリシーが必要です。

* _"What are my recent emails about the Professional Claw Hammer stock issue at Seattle?"_ → **Work IQ** から返るはず
* _"Which Zava job roles are responsible for inventory strategy and budget oversight?"_ → `hrdocs` から返るはず

> ⏳ Work IQ により retrieval の latency が大きくなるため、応答までに 1〜2 分かかります。retrieval リクエストでは `max_runtime_in_seconds` を指定し、knowledge base がソースからの結果を待つ時間を延ばしています。

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("Check my recent work emails for messages about the Professional Claw Hammer. "
            "Summarize what colleagues are saying and what actions have been requested. " 
            "Which Zava job roles are responsible for inventory strategy and budget oversight?")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)
display(Markdown(result.response[0].content[0].text))


### activity log を確認する

今回の activity log には、Work IQ への呼び出しに対応する `workIQ` ステップと、各検索インデックスへのクエリごとの `searchIndex` ステップが表示されます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

Work IQ knowledge source については、references の `type` が `workIQ` になります。`sourceData` には Work IQ エージェントが合成した回答が入った `extracts` が含まれ、各 reference には参照されたメールやドキュメントなどへのリンクとなる URL を持つ `attributions` が含まれます。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上に表示された references を見て、次を特定してみましょう。

1. **メールスレッドの要約** に関する質問にはどの knowledge source が答えましたか？
2. **職務ロール** に関する質問にはどの knowledge source が答えましたか？

## ✅ ミッション達成

**構築したもの:**

* ✓ **Work IQ Knowledge Source**: 実際の Microsoft 365 データを knowledge base に接続するソース。KB はクエリ時にメール、Teams メッセージ、カレンダーのコンテキストを取得できます。
* ✓ **3 ソースの Knowledge Base**: HR docs、health docs、ライブの M365 職場データにまたがる単一の KB。エージェントが各サブクエスチョンを適切なソースへルーティングします。
* ✓ **受信箱とポリシーのハイブリッド取得**: Seattle の在庫切れに関するメールスレッドは Work IQ が抽出し、予算管理の質問には HR インデックスが回答しました。

➡️ 次へ: [Part 5: Work IQ と Fabric IQ を組み合わせる](part5-work-iq-fabric-iq-to-kb.ipynb)。